In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import os, re, sys, hashlib, shutil, datetime as dt

# --------- Dependências críticas ---------
missing = []
try:
    import pandas as pd
except Exception:
    pd = None
    missing.append("pandas")
try:
    from PyPDF2 import PdfReader
except Exception:
    PdfReader = None
    missing.append("PyPDF2")
try:
    import openpyxl  # apenas para o writer do pandas
except Exception:
    missing.append("openpyxl")

if missing:
    print("⚠️ Dependências ausentes:", ", ".join(missing))
    print("   Instale no Anaconda Prompt / terminal:")
    if "pandas" in missing: print("   pip install pandas")
    if "openpyxl" in missing: print("   pip install openpyxl")
    if "PyPDF2" in missing: print("   pip install PyPDF2")

# ======================================================================
# Parâmetros fixos do seu ambiente (ajuste se quiser)
# ======================================================================
BASE_PROJETOS = r"C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\NFS\2025"

PROJETOS = [
    "ESPECIAL", "LAUDO DE RISCO", "LAUDO DE RISCO CD", "LAUDO DE RISCO MATRIZ",
    "LOJAS NOVAS", "MISSOES HIGIENICAS", "OUTROS", "PROJETO ESPECIAL",
    "RESIZE", "RETROFIT", "VIRADA DE BANDEIRA",
]

LOOKBACK_DAYS_DEFAULT = 15  # usado se você optar por filtrar por data
DIAGNOSTICO = True

# ======================================================================
# UI Terminal
# ======================================================================
def select_projeto_menu(opcoes: list) -> str:
    print("\n📁 Selecione o PROJETO:")
    for i, nome in enumerate(opcoes, start=1):
        print(f" {i}. {nome}")
    print(" 0. Cancelar")
    while True:
        try:
            op = int(input("Digite o número da opção: ").strip())
            if op == 0:
                return None
            if 1 <= op <= len(opcoes):
                projeto = opcoes[op - 1]
                print(f"✅ Selecionado: {projeto}")
                return projeto
        except ValueError:
            pass
        print("Opção inválida. Tente novamente.")

def ask_fornecedor_nome() -> str:
    nome = input("\nDigite o NOME exato do FORNECEDOR (ex.: LUMICENTER): ").strip()
    while not nome:
        nome = input("Digite o NOME do FORNECEDOR: ").strip()
    return nome

def ask_pasta_origem() -> str:
    while True:
        p = input("\nDigite o CAMINHO da PASTA onde estão os PDFs (ex.: C:\\Temp\\NFs): ").strip().strip('"')
        if os.path.isdir(p):
            return p
        print("❌ Caminho inválido. Tente novamente.")

def ask_yesno(msg: str, default_yes=True) -> bool:
    sufixo = "[S/n]" if default_yes else "[s/N]"
    while True:
        resp = input(f"{msg} {sufixo}: ").strip().lower()
        if not resp:
            return default_yes
        if resp in ("s", "sim", "y", "yes"):
            return True
        if resp in ("n", "nao", "não", "no"):
            return False
        print("Resposta inválida. Digite S ou N.")

def ask_int(msg: str, padrao: int) -> int:
    val = input(f"{msg} (padrão {padrao}): ").strip()
    if not val:
        return padrao
    try:
        return int(val)
    except:
        print("Valor inválido, usando padrão.")
        return padrao

# ======================================================================
# Utils locais
# ======================================================================
def _ensure_dirs(path): os.makedirs(path, exist_ok=True)

def _sanitize_filename(name: str) -> str:
    name = (name or "").strip().replace("\n", " ").replace("\r", " ")
    name = re.sub(r'[<>:"/\\|?*]', "_", name)
    name = re.sub(r"\s{2,}", " ", name)
    return name

def _md5_file(path: str) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def _normalize_text(text: str) -> str:
    if not text:
        return ""
    t = re.sub(r"[\n\r\t]+", " ", text)
    t = re.sub(r"\s{2,}", " ", t)
    return t.upper()

def _parse_brl_to_float(v):
    if v is None:
        return None
    s = str(v)
    s = re.sub(r"[Rr]\$\s*", "", s)
    s = s.replace(".", "TEMP").replace(",", ".").replace("TEMP", "")
    s = re.sub(r"[^\d\.]", "", s)
    try:
        return float(s)
    except:
        return None

# ======================================================================
# CNPJ & Loja
# ======================================================================
def _cnpj_digits(s: str) -> str:
    d = re.sub(r"\D", "", s or "")
    return d if len(d) == 14 else ""

def _cnpj_is_valid(d: str) -> bool:
    d = _cnpj_digits(d)
    if len(d) != 14 or len(set(d)) == 1:
        return False
    def dv_calc(digs, pesos):
        soma = sum(int(digs[i]) * pesos[i] for i in range(len(pesos)))
        r = soma % 11
        return '0' if r < 2 else str(11 - r)
    base = d[:12]
    dv1 = dv_calc(base, [5,4,3,2,9,8,7,6,5,4,3,2])
    dv2 = dv_calc(base + dv1, [6,5,4,3,2,9,8,7,6,5,4,3,2])
    return d[-2:] == dv1 + dv2

def _derive_loja_from_cnpj(cnpj_raw: str, origem_cnpj: str) -> tuple:
    d = _cnpj_digits(cnpj_raw)
    if not d:
        return (None, origem_cnpj)
    seq = d[8:12]
    origem_loja = origem_cnpj
    if d.startswith("04"):
        loja = "7" + seq[1:]; origem_loja += " + Regra IMIFARMA"
    elif d.startswith("06"):
        loja = seq.lstrip("0") or "0"; origem_loja += " + Regra Pague Menos"
    else:
        loja = seq.lstrip("0") or seq
    return loja, origem_loja

# ======================================================================
# Extratores
# ======================================================================
def extrair_vencimento_de_texto(texto: str) -> str or None:
    if not texto: return None
    t = _normalize_text(texto).upper()
    m = re.search(r"\b(VENC(?:IMENTO)?|VENCTO|VCTO)\b[^0-9]{0,10}(\d{2}/\d{2}/\d{4})", t)
    if m: return m.group(2)
    m2 = re.search(r"\b(\d{2}/\d{2}/\d{4})\b", t)
    return m2.group(1) if m2 else None

def _find_oc_candidates(text_upper: str) -> list:
    oc_pattern = r"\b(45\d{8,10})\b"
    cands = []
    labels = ["ORDEM DE COMPRA", "OC", "PEDIDO", "PEDIDO CLIENTE", "DADOS ADICIONAIS", "INFORMAÇÕES COMPLEMENTARES"]
    for lbl in labels:
        m_lbl = re.search(lbl, text_upper)
        if m_lbl:
            start = max(0, m_lbl.start()-2000)
            bloco = text_upper[start: m_lbl.start()+4000]
            for m in re.finditer(oc_pattern, bloco):
                cands.append((start + m.start(), m.group(1), 5))
    for m in re.finditer(oc_pattern, text_upper):
        cands.append((m.start(), m.group(1), 1))
    seen, uniq = set(), []
    for pos, oc, score in cands:
        if oc not in seen:
            seen.add(oc); uniq.append((pos, oc, score))
    return uniq

def extrair_dados_pdf(caminho_pdf, debug=False):
    if PdfReader is None:
        return None, None, None, None, None, None, {}
    try:
        reader = PdfReader(caminho_pdf)
        texto_completo = "\n".join([p.extract_text() or "" for p in reader.pages])
        t_upper = texto_completo.upper()

        # --- CNPJ E LOJA ---
        cnpj_dest = None
        m_cnpj = re.search(r"(\d{2}\.\d{3}\.\d{3}/\d{4}\-\d{2})", t_upper)
        if m_cnpj:
            cnpj_dest = _cnpj_digits(m_cnpj.group(1))
        loja, origem_loja = _derive_loja_from_cnpj(cnpj_dest, "CNPJ")

        if not loja:
            m_lj = re.search(r"(?:LOJA|LJ|FL)[^\d]{0,10}(\d{1,5})\b", t_upper)
            if m_lj:
                loja = m_lj.group(1)

        # --- NÚMERO DA NF (evitar capturar DDD/telefone como NF) ---
        nf = None
        # 1) Padrão específico com Inscrição Municipal
        m_insc = re.search(r"INSCRIÇÃO\s+MUNICIPAL:\s*\d{8,}(\d{4,7})\b", t_upper)
        if m_insc:
            nf = m_insc.group(1)
        # 2) Rótulos comuns, filtrando "(98)" e "TELEFONE"
        if not nf:
            padroes = [
                r"N[ÚU]MERO\s+DA\s+NOTA[\r\n\s:]*(\d+)",
                r"N[ÚU]MERO\s+DA\s+NFS?-?E[\r\n\s:]*(\d+)",
                r"NF-E\s+N[º°\.\s]*(\d+)"
            ]
            for p in padroes:
                for m in re.finditer(p, t_upper):
                    cand = m.group(1)
                    contexto = t_upper[max(0, m.start()-5):m.end()+20]
                    if "(98)" in contexto or "TELEFONE" in contexto:
                        continue
                    if 1 < len(cand) < 10:
                        nf = cand
                        break
                if nf:
                    break

        # --- OUTROS CAMPOS ---
        serie = (re.search(r"S[ÉE]RIE[^\dA-Z]{0,10}(\d+|[A-Z]+)\b", t_upper) or [None, None])[1]
        valor = (re.search(r"VALOR\s*TOTAL[^\d]{0,20}([\d\.,]{4,})", t_upper) or [None, None])[1]
        chave = (re.search(r"CHAVE[^\d]{0,20}(\d{44})", t_upper) or [None, None])[1]
        oc_cands = _find_oc_candidates(t_upper)
        oc = max(oc_cands, key=lambda x: x[2])[1] if oc_cands else None
        data_emissao = (re.search(r"(?:DATA\s*(?:DE\s*|E\s*HORA\s*(?:DA\s*|DE\s*))?EMISS[ÃA]O)[^\d]{0,60}(\d{2}/\d{2}/\d{4})", t_upper) or [None, None])[1]

        extras = {
            "Valor Total": valor,
            "Valor Total Num": _parse_brl_to_float(valor),
            "ordem_compra": oc,
            "pedido": oc,
            "data_emissao": data_emissao,
            "destinatario_cnpj": cnpj_dest
        }
        return nf, loja, "PDF", origem_loja, serie, chave, extras
    except Exception as e:
        if debug: print(f"Erro ao ler {caminho_pdf}: {e}")
        return None, None, None, None, None, None, {}

# ======================================================================
# Listagem de PDFs a partir de uma pasta
# ======================================================================
def listar_pdfs(pasta_origem: str, recursivo=True, filtrar_por_data=True, lookback_days=LOOKBACK_DAYS_DEFAULT):
    pdfs = []
    dt_limite = dt.datetime.now() - dt.timedelta(days=lookback_days)
    def dentro_periodo(path):
        if not filtrar_por_data:
            return True
        try:
            mtime = dt.datetime.fromtimestamp(os.path.getmtime(path))
            return mtime >= dt_limite
        except:
            return True

    if recursivo:
        for root, _, files in os.walk(pasta_origem):
            for fn in files:
                if fn.lower().endswith(".pdf"):
                    full = os.path.join(root, fn)
                    if dentro_periodo(full):
                        pdfs.append(full)
    else:
        for fn in os.listdir(pasta_origem):
            full = os.path.join(pasta_origem, fn)
            if os.path.isfile(full) and fn.lower().endswith(".pdf") and dentro_periodo(full):
                pdfs.append(full)
    return pdfs

# ======================================================================
# MAIN
# ======================================================================
def main():
    # 1) Seleções básicas
    projeto = select_projeto_menu(PROJETOS)
    if not projeto:
        return
    fornecedor = ask_fornecedor_nome()

    # 2) Pastas
    pasta_origem = ask_pasta_origem()
    recursivo = ask_yesno("Varrer SUBPASTAS?", default_yes=True)
    usar_filtro_data = ask_yesno(f"Filtrar pelos últimos {LOOKBACK_DAYS_DEFAULT} dias?", default_yes=True)
    lookback_days = LOOKBACK_DAYS_DEFAULT if usar_filtro_data else 99999

    pasta_projeto = os.path.join(BASE_PROJETOS, projeto)
    pasta_fornecedor = os.path.join(pasta_projeto, fornecedor)
    if not os.path.isdir(pasta_projeto):
        print(f"❌ Pasta do projeto não existe: {pasta_projeto}")
        return
    _ensure_dirs(pasta_fornecedor)

    # 3) Coleta PDFs na pasta informada
    pdf_paths = listar_pdfs(pasta_origem, recursivo=recursivo, filtrar_por_data=usar_filtro_data, lookback_days=lookback_days)
    print(f"\n📥 PDFs encontrados: {len(pdf_paths)}")
    if DIAGNOSTICO and pdf_paths[:5]:
        print("Exemplos:", *pdf_paths[:5], sep="\n - ")

    # 4) Processamento
    base_registros = []
    for src_path in pdf_paths:
        try:
            nf, loja, _, origem_lo, serie, chave, extras = extrair_dados_pdf(src_path)

            novo_nome = f"NF {nf} - LJ {loja}.pdf" if nf and loja else f"NF {nf or 'SEM_NUM'}.pdf"
            novo_nome = _sanitize_filename(novo_nome)
            destino = os.path.join(pasta_fornecedor, novo_nome)

            status = "OK"
            try:
                if os.path.exists(destino):
                    # Evita sobrescrever: acrescenta hash curto
                    hash5 = _md5_file(src_path)[:5]
                    base, ext = os.path.splitext(novo_nome)
                    destino = os.path.join(pasta_fornecedor, f"{base}_{hash5}{ext}")
                shutil.move(src_path, destino)
            except Exception as e:
                status = f"Erro ao mover: {e}"

            base_registros.append({
                "Projeto": projeto,
                "Fornecedor": fornecedor,
                "NF": nf,
                "Loja": loja,
                "Serie": serie,
                "Chave": chave,
                "Valor Total": extras.get("Valor Total"),
                "Valor Total Num": extras.get("Valor Total Num"),
                "Pedido": extras.get("pedido"),
                "Data Emissao": extras.get("data_emissao"),
                "CNPJ Destinatario": extras.get("destinatario_cnpj"),
                "OrigemArquivo": src_path,
                "DestinoArquivo": destino,
                "Status": status
            })
        except Exception as e:
            base_registros.append({
                "Projeto": projeto, "Fornecedor": fornecedor, "NF": None, "Loja": None,
                "Serie": None, "Chave": None, "Valor Total": None, "Valor Total Num": None,
                "Pedido": None, "Data Emissao": None, "CNPJ Destinatario": None,
                "OrigemArquivo": src_path, "DestinoArquivo": None, "Status": f"Erro geral: {e}"
            })

    # 5) Exporta Excel
    if base_registros and pd:
        df = pd.DataFrame(base_registros)
        hoje = dt.datetime.now().strftime("%Y-%m-%d")
        excel_path = os.path.join(pasta_projeto, f"BASE_NFs_{fornecedor}_{hoje}.xlsx")
        try:
            df.to_excel(excel_path, index=False)
            print(f"\n✅ Excel gerado: {excel_path}")
        except Exception as e:
            print(f"❌ Falha ao salvar Excel: {e}")
    else:
        print("⚠️ Nenhum registro para exportar ou pandas não instalado.")

    # 6) Resumo
    ok = sum(1 for r in base_registros if r.get("Status") == "OK")
    falhas = sum(1 for r in base_registros if r.get("Status") != "OK")
    print(f"\n📊 Resumo: OK={ok}  Falhas={falhas}  Total={len(base_registros)}")
    if falhas:
        print("Dicas:")
        print(" - Verifique permissões da pasta de destino.")
        print(" - Feche arquivos PDF abertos para permitir o move.")
        print(" - Alguns PDFs podem não ter texto selecionável (PDF imagem).")

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\nInterrompido pelo usuário.")


📁 Selecione o PROJETO:
 1. ESPECIAL
 2. LAUDO DE RISCO
 3. LAUDO DE RISCO CD
 4. LAUDO DE RISCO MATRIZ
 5. LOJAS NOVAS
 6. MISSOES HIGIENICAS
 7. OUTROS
 8. PROJETO ESPECIAL
 9. RESIZE
 10. RETROFIT
 11. VIRADA DE BANDEIRA
 0. Cancelar


In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import os, re, sys, hashlib, shutil, datetime as dt

# --------- Dependências críticas ---------
missing = []
try:
    import pandas as pd
except Exception:
    pd = None
    missing.append("pandas")
try:
    from PyPDF2 import PdfReader
except Exception:
    PdfReader = None
    missing.append("PyPDF2")
try:
    import openpyxl  # apenas para o writer do pandas
except Exception:
    missing.append("openpyxl")

if missing:
    print("⚠️ Dependências ausentes:", ", ".join(missing))
    print("   Instale no Anaconda Prompt / terminal:")
    if "pandas" in missing: print("   pip install pandas")
    if "openpyxl" in missing: print("   pip install openpyxl")
    if "PyPDF2" in missing: print("   pip install PyPDF2")

# ======================================================================
# Parâmetros fixos do seu ambiente (ajuste se quiser)
# ======================================================================
BASE_PROJETOS = r"C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\NFS\2025"

PROJETOS = [
    "ESPECIAL", "LAUDO DE RISCO", "LAUDO DE RISCO CD", "LAUDO DE RISCO MATRIZ",
    "LOJAS NOVAS", "MISSOES HIGIENICAS", "OUTROS", "PROJETO ESPECIAL",
    "RESIZE", "RETROFIT", "VIRADA DE BANDEIRA",
]

LOOKBACK_DAYS_DEFAULT = 15  # usado se você optar por filtrar por data
DIAGNOSTICO = True

# ======================================================================
# UI Terminal
# ======================================================================
def select_projeto_menu(opcoes: list) -> str:
    print("\n📁 Selecione o PROJETO:")
    for i, nome in enumerate(opcoes, start=1):
        print(f" {i}. {nome}")
    print(" 0. Cancelar")
    while True:
        try:
            op = int(input("Digite o número da opção: ").strip())
            if op == 0:
                return None
            if 1 <= op <= len(opcoes):
                projeto = opcoes[op - 1]
                print(f"✅ Selecionado: {projeto}")
                return projeto
        except ValueError:
            pass
        print("Opção inválida. Tente novamente.")

def ask_fornecedor_nome() -> str:
    nome = input("\nDigite o NOME exato do FORNECEDOR (ex.: LUMICENTER): ").strip()
    while not nome:
        nome = input("Digite o NOME do FORNECEDOR: ").strip()
    return nome

def ask_pasta_origem() -> str:
    while True:
        p = input("\nDigite o CAMINHO da PASTA onde estão os PDFs (ex.: C:\\Temp\\NFs): ").strip().strip('"')
        if os.path.isdir(p):
            return p
        print("❌ Caminho inválido. Tente novamente.")

def ask_yesno(msg: str, default_yes=True) -> bool:
    sufixo = "[S/n]" if default_yes else "[s/N]"
    while True:
        resp = input(f"{msg} {sufixo}: ").strip().lower()
        if not resp:
            return default_yes
        if resp in ("s", "sim", "y", "yes"):
            return True
        if resp in ("n", "nao", "não", "no"):
            return False
        print("Resposta inválida. Digite S ou N.")

def ask_int(msg: str, padrao: int) -> int:
    val = input(f"{msg} (padrão {padrao}): ").strip()
    if not val:
        return padrao
    try:
        return int(val)
    except:
        print("Valor inválido, usando padrão.")
        return padrao

# ======================================================================
# Utils locais
# ======================================================================
def _ensure_dirs(path): os.makedirs(path, exist_ok=True)

def _sanitize_filename(name: str) -> str:
    name = (name or "").strip().replace("\n", " ").replace("\r", " ")
    name = re.sub(r'[<>:"/\\|?*]', "_", name)
    name = re.sub(r"\s{2,}", " ", name)
    return name

def _md5_file(path: str) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def _normalize_text(text: str) -> str:
    if not text:
        return ""
    t = re.sub(r"[\n\r\t]+", " ", text)
    t = re.sub(r"\s{2,}", " ", t)
    return t.upper()

def _parse_brl_to_float(v):
    if v is None:
        return None
    s = str(v)
    s = re.sub(r"[Rr]\$\s*", "", s)
    s = s.replace(".", "TEMP").replace(",", ".").replace("TEMP", "")
    s = re.sub(r"[^\d\.]", "", s)
    try:
        return float(s)
    except:
        return None

# ======================================================================
# CNPJ & Loja
# ======================================================================
def _cnpj_digits(s: str) -> str:
    d = re.sub(r"\D", "", s or "")
    return d if len(d) == 14 else ""

def _cnpj_is_valid(d: str) -> bool:
    d = _cnpj_digits(d)
    if len(d) != 14 or len(set(d)) == 1:
        return False
    def dv_calc(digs, pesos):
        soma = sum(int(digs[i]) * pesos[i] for i in range(len(pesos)))
        r = soma % 11
        return '0' if r < 2 else str(11 - r)
    base = d[:12]
    dv1 = dv_calc(base, [5,4,3,2,9,8,7,6,5,4,3,2])
    dv2 = dv_calc(base + dv1, [6,5,4,3,2,9,8,7,6,5,4,3,2])
    return d[-2:] == dv1 + dv2

def _derive_loja_from_cnpj(cnpj_raw: str, origem_cnpj: str) -> tuple:
    d = _cnpj_digits(cnpj_raw)
    if not d:
        return (None, origem_cnpj)
    seq = d[8:12]  # 4 dígitos do estabelecimento
    origem_loja = origem_cnpj
    if d.startswith("04"):
        loja = "7" + seq[1:]; origem_loja += " + Regra IMIFARMA"
    elif d.startswith("06"):
        loja = seq.lstrip("0") or "0"; origem_loja += " + Regra Pague Menos"
    else:
        loja = seq.lstrip("0") or seq
    return loja, origem_loja

# ======================================================================
# Extratores
# ======================================================================
def extrair_vencimento_de_texto(texto: str) -> str or None:
    if not texto: return None
    t = _normalize_text(texto).upper()
    m = re.search(r"\b(VENC(?:IMENTO)?|VENCTO|VCTO)\b[^0-9]{0,10}(\d{2}/\d{2}/\d{4})", t)
    if m: return m.group(2)
    m2 = re.search(r"\b(\d{2}/\d{2}/\d{4})\b", t)
    return m2.group(1) if m2 else None

def _find_oc_candidates(text_upper: str) -> list:
    oc_pattern = r"\b(45\d{8,10})\b"
    cands = []
    labels = ["ORDEM DE COMPRA", "OC", "PEDIDO", "PEDIDO CLIENTE", "DADOS ADICIONAIS", "INFORMAÇÕES COMPLEMENTARES"]
    for lbl in labels:
        m_lbl = re.search(lbl, text_upper)
        if m_lbl:
            start = max(0, m_lbl.start()-2000)
            bloco = text_upper[start: m_lbl.start()+4000]
            for m in re.finditer(oc_pattern, bloco):
                cands.append((start + m.start(), m.group(1), 5))
    for m in re.finditer(oc_pattern, text_upper):
        cands.append((m.start(), m.group(1), 1))
    seen, uniq = set(), []
    for pos, oc, score in cands:
        if oc not in seen:
            seen.add(oc); uniq.append((pos, oc, score))
    return uniq

# ======================================================================
# Leitura de PDF + regras de extração (AJUSTADO)
# ======================================================================
def extrair_dados_pdf(caminho_pdf, debug=False):
    if PdfReader is None:
        return None, None, None, None, None, None, {}
    try:
        reader = PdfReader(caminho_pdf)
        texto_completo = "\n".join([p.extract_text() or "" for p in reader.pages])
        t_upper = texto_completo.upper()

        # ---------- CNPJ DESTINATÁRIO (corrigido) ----------
        def _pick_cnpj_dest(text_upper: str) -> str:
            # Encontra todos os CNPJs (formatados)
            cnpj_iter = list(re.finditer(r"\d{2}\.\d{3}\.\d{3}/\d{4}\-\d{2}", text_upper))
            if not cnpj_iter:
                return ""
            scored = []
            for m in cnpj_iter:
                start, end = m.start(), m.end()
                win = text_upper[max(0, start-250): min(len(text_upper), end+250)]
                score = 0
                # Sinais de DESTINATÁRIO/ENTREGA/IDENTIFICAÇÃO DO RECEBEDOR
                if re.search(r"DESTINAT[ÁA]RIO|LOCAL\s+DE\s+ENTREGA|RECEBEDOR|NOME/RAZ[ÃA]O\s+SOCIAL", win):
                    score += 3
                # Sinais de Pague Menos / Imifarma
                if re.search(r"PAGUE\s+MENOS|IMIFARMA|EMPREENDE?NDIMENTOS\s+PAGUE\s+MENOS", win):
                    score += 2
                # Penaliza EMITENTE e blocos de cabeçalho do emitente
                if re.search(r"EMITENTE|IDENTIFICA[ÇC][ÃA]O\s+DO\s+EMITENTE", win):
                    score -= 3
                # Pequeno bônus se o CNPJ for '06....' (holding/grupo)
                if _cnpj_digits(m.group()).startswith("06"):
                    score += 1
                scored.append((score, m.group(), start))
            scored.sort(key=lambda x: (-x[0], x[2]))
            return _cnpj_digits(scored[0][1]) if scored else ""

        cnpj_dest = _pick_cnpj_dest(t_upper)

        # Deriva loja a partir do CNPJ do destinatário (sua regra original)
        loja, origem_loja = _derive_loja_from_cnpj(cnpj_dest, "CNPJ")

        # ---------- Fallback por 'LJ' no texto ----------
        # Captura LJ sem/ com espaço (LJ269 ou LJ 269) e também LOJA/FL
        m_lj_any = re.search(r"\b(?:LOJA|LJ|FL)\b[^\d]{0,10}(\d{1,5})\b", t_upper)
        if not loja and m_lj_any:
            loja = m_lj_any.group(1)
            origem_loja = (origem_loja or "CNPJ") + " + Fallback LJ"
        else:
            # Se loja do CNPJ saiu muito curta (ex.: '5') e há LJ de 2+ dígitos no texto, priorize o LJ
            m_lj2 = re.search(r"\bLJ[^\d]{0,2}(\d{2,5})\b", t_upper)
            if loja and loja.isdigit() and int(loja) < 10 and m_lj2:
                loja = m_lj2.group(1)
                origem_loja = (origem_loja or "CNPJ") + " + Fallback LJ"

        # ---------- NÚMERO DA NF ----------
        nf = None
        # 1) Padrão específico com Inscrição Municipal
        m_insc = re.search(r"INSCRIÇÃO\s+MUNICIPAL:\s*\d{8,}(\d{4,7})\b", t_upper)
        if m_insc:
            nf = m_insc.group(1)
        # 2) Rótulos comuns, filtrando "(98)" e "TELEFONE"
        if not nf:
            padroes = [
                r"N[ÚU]MERO\s+DA\s+NOTA[\r\n\s:]*(\d+)",
                r"N[ÚU]MERO\s+DA\s+NFS?-?E[\r\n\s:]*(\d+)",
                r"NF-E\s+N[º°\.\s]*(\d+)"
            ]
            for p in padroes:
                for m in re.finditer(p, t_upper):
                    cand = m.group(1)
                    contexto = t_upper[max(0, m.start()-5):m.end()+20]
                    if "(98)" in contexto or "TELEFONE" in contexto:
                        continue
                    if 1 < len(cand) < 10:
                        nf = cand
                        break
                if nf:
                    break

        # --- OUTROS CAMPOS ---
        serie = (re.search(r"S[ÉE]RIE[^\dA-Z]{0,10}(\d+|[A-Z]+)\b", t_upper) or [None, None])[1]
        valor = (re.search(r"VALOR\s*TOTAL[^\d]{0,20}([\d\.,]{4,})", t_upper) or [None, None])[1]
        chave = (re.search(r"CHAVE[^\d]{0,20}(\d{44})", t_upper) or [None, None])[1]
        oc_cands = _find_oc_candidates(t_upper)
        oc = max(oc_cands, key=lambda x: x[2])[1] if oc_cands else None
        data_emissao = (re.search(r"(?:DATA\s*(?:DE\s*|E\s*HORA\s*(?:DA\s*|DE\s*))?EMISS[ÃA]O)[^\d]{0,60}(\d{2}/\d{2}/\d{4})", t_upper) or [None, None])[1]

        if DIAGNOSTICO:
            print(f" - CNPJ Destinatário: {cnpj_dest or 'N/D'} | Loja derivada: {loja or 'N/D'} | Origem Loja: {origem_loja or 'N/D'}")

        extras = {
            "Valor Total": valor,
            "Valor Total Num": _parse_brl_to_float(valor),
            "ordem_compra": oc,
            "pedido": oc,
            "data_emissao": data_emissao,
            "destinatario_cnpj": cnpj_dest
        }
        return nf, loja, "PDF", origem_loja, serie, chave, extras
    except Exception as e:
        if debug: print(f"Erro ao ler {caminho_pdf}: {e}")
        return None, None, None, None, None, None, {}

# ======================================================================
# Listagem de PDFs a partir de uma pasta
# ======================================================================
def listar_pdfs(pasta_origem: str, recursivo=True, filtrar_por_data=True, lookback_days=LOOKBACK_DAYS_DEFAULT):
    pdfs = []
    dt_limite = dt.datetime.now() - dt.timedelta(days=lookback_days)
    def dentro_periodo(path):
        if not filtrar_por_data:
            return True
        try:
            mtime = dt.datetime.fromtimestamp(os.path.getmtime(path))
            return mtime >= dt_limite
        except:
            return True

    if recursivo:
        for root, _, files in os.walk(pasta_origem):
            for fn in files:
                if fn.lower().endswith(".pdf"):
                    full = os.path.join(root, fn)
                    if dentro_periodo(full):
                        pdfs.append(full)
    else:
        for fn in os.listdir(pasta_origem):
            full = os.path.join(pasta_origem, fn)
            if os.path.isfile(full) and fn.lower().endswith(".pdf") and dentro_periodo(full):
                pdfs.append(full)
    return pdfs

# ======================================================================
# MAIN
# ======================================================================
def main():
    # 1) Seleções básicas
    projeto = select_projeto_menu(PROJETOS)
    if not projeto:
        return
    fornecedor = ask_fornecedor_nome()

    # 2) Pastas
    pasta_origem = ask_pasta_origem()
    recursivo = ask_yesno("Varrer SUBPASTAS?", default_yes=True)
    usar_filtro_data = ask_yesno(f"Filtrar pelos últimos {LOOKBACK_DAYS_DEFAULT} dias?", default_yes=True)
    lookback_days = LOOKBACK_DAYS_DEFAULT if usar_filtro_data else 99999

    pasta_projeto = os.path.join(BASE_PROJETOS, projeto)
    pasta_fornecedor = os.path.join(pasta_projeto, fornecedor)
    if not os.path.isdir(pasta_projeto):
        print(f"❌ Pasta do projeto não existe: {pasta_projeto}")
        return
    _ensure_dirs(pasta_fornecedor)

    # 3) Coleta PDFs na pasta informada
    pdf_paths = listar_pdfs(pasta_origem, recursivo=recursivo, filtrar_por_data=usar_filtro_data, lookback_days=lookback_days)
    print(f"\n📥 PDFs encontrados: {len(pdf_paths)}")
    if DIAGNOSTICO and pdf_paths[:5]:
        print("Exemplos:", *pdf_paths[:5], sep="\n - ")

    # 4) Processamento
    base_registros = []
    for src_path in pdf_paths:
        try:
            nf, loja, _, origem_lo, serie, chave, extras = extrair_dados_pdf(src_path)

            novo_nome = f"NF {nf} - LJ {loja}.pdf" if nf and loja else f"NF {nf or 'SEM_NUM'}.pdf"
            novo_nome = _sanitize_filename(novo_nome)
            destino = os.path.join(pasta_fornecedor, novo_nome)

            status = "OK"
            try:
                if os.path.exists(destino):
                    # Evita sobrescrever: acrescenta hash curto
                    hash5 = _md5_file(src_path)[:5]
                    base, ext = os.path.splitext(novo_nome)
                    destino = os.path.join(pasta_fornecedor, f"{base}_{hash5}{ext}")
                shutil.move(src_path, destino)
            except Exception as e:
                status = f"Erro ao mover: {e}"

            base_registros.append({
                "Projeto": projeto,
                "Fornecedor": fornecedor,
                "NF": nf,
                "Loja": loja,
                "Serie": serie,
                "Chave": chave,
                "Valor Total": extras.get("Valor Total"),
                "Valor Total Num": extras.get("Valor Total Num"),
                "Pedido": extras.get("pedido"),
                "Data Emissao": extras.get("data_emissao"),
                "CNPJ Destinatario": extras.get("destinatario_cnpj"),
                "OrigemArquivo": src_path,
                "DestinoArquivo": destino,
                "Status": status
            })
        except Exception as e:
            base_registros.append({
                "Projeto": projeto, "Fornecedor": fornecedor, "NF": None, "Loja": None,
                "Serie": None, "Chave": None, "Valor Total": None, "Valor Total Num": None,
                "Pedido": None, "Data Emissao": None, "CNPJ Destinatario": None,
                "OrigemArquivo": src_path, "DestinoArquivo": None, "Status": f"Erro geral: {e}"
            })

    # 5) Exporta Excel
    if base_registros and pd:
        df = pd.DataFrame(base_registros)
        hoje = dt.datetime.now().strftime("%Y-%m-%d")
        excel_path = os.path.join(pasta_projeto, f"BASE_NFs_{fornecedor}_{hoje}.xlsx")
        try:
            df.to_excel(excel_path, index=False)
            print(f"\n✅ Excel gerado: {excel_path}")
        except Exception as e:
            print(f"❌ Falha ao salvar Excel: {e}")
    else:
        print("⚠️ Nenhum registro para exportar ou pandas não instalado.")

    # 6) Resumo
    ok = sum(1 for r in base_registros if r.get("Status") == "OK")
    falhas = sum(1 for r in base_registros if r.get("Status") != "OK")
    print(f"\n📊 Resumo: OK={ok}  Falhas={falhas}  Total={len(base_registros)}")
    if falhas:
        print("Dicas:")
        print(" - Verifique permissões da pasta de destino.")
        print(" - Feche arquivos PDF abertos para permitir o move.")
        print(" - Alguns PDFs podem não ter texto selecionável (PDF imagem).")

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\nInterrompido pelo usuário.")


📁 Selecione o PROJETO:
 1. ESPECIAL
 2. LAUDO DE RISCO
 3. LAUDO DE RISCO CD
 4. LAUDO DE RISCO MATRIZ
 5. LOJAS NOVAS
 6. MISSOES HIGIENICAS
 7. OUTROS
 8. PROJETO ESPECIAL
 9. RESIZE
 10. RETROFIT
 11. VIRADA DE BANDEIRA
 0. Cancelar


Digite o número da opção:  5


✅ Selecionado: LOJAS NOVAS



Digite o NOME exato do FORNECEDOR (ex.: LUMICENTER):  HTB

Digite o CAMINHO da PASTA onde estão os PDFs (ex.: C:\Temp\NFs):  C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\NFS\2025\LOJAS NOVAS\HTB\24.03.2026
Varrer SUBPASTAS? [S/n]:  s
Filtrar pelos últimos 15 dias? [S/n]:  s



📥 PDFs encontrados: 6
Exemplos:
 - C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\NFS\2025\LOJAS NOVAS\HTB\24.03.2026\NFS 15706 - COD03.pdf
 - C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\NFS\2025\LOJAS NOVAS\HTB\24.03.2026\NFS 15707 - APG02.pdf
 - C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\NFS\2025\LOJAS NOVAS\HTB\24.03.2026\NFS 15708 - CGD24.pdf
 - C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\NFS\2025\LOJAS NOVAS\HTB\24.03.2026\NFS 15709 - IGU04.pdf
 - C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\NFS\2025\LOJAS NOVAS\HTB\24.03.2026\NFS 15710 - PAR08.pdf
 - CNPJ Destinatário: 06626253133461 | Loja derivada: 1334 | Origem Loja: CNPJ + Regra Pague Menos
 - CNPJ Destinatário: 06626253149970 | Loja derivada: 1499 | Origem Loja: CNPJ + Regra Pague Menos
 - CNPJ Destinatário: 06626253149112 | Loja derivada: 1491 | Origem Loja: CNPJ + Regra Pague Menos
 - CNPJ Destinatário: 06626253052038 | Loja derivada: 5

In [2]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import os, re, sys, hashlib, shutil, datetime as dt

# --------- Dependências críticas ---------
missing = []
try:
    import pandas as pd
except Exception:
    pd = None
    missing.append("pandas")
try:
    from PyPDF2 import PdfReader
except Exception:
    PdfReader = None
    missing.append("PyPDF2")
try:
    import openpyxl  # apenas para o writer do pandas
except Exception:
    missing.append("openpyxl")

if missing:
    print("⚠️ Dependências ausentes:", ", ".join(missing))
    print("   Instale no Anaconda Prompt / terminal:")
    if "pandas" in missing: print("   pip install pandas")
    if "openpyxl" in missing: print("   pip install openpyxl")
    if "PyPDF2" in missing: print("   pip install PyPDF2")

# ======================================================================
# Parâmetros fixos do seu ambiente (ajuste se quiser)
# ======================================================================
BASE_PROJETOS = r"C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\NFS\2025"

PROJETOS = [
    "ESPECIAL", "LAUDO DE RISCO", "LAUDO DE RISCO CD", "LAUDO DE RISCO MATRIZ",
    "LOJAS NOVAS", "MISSOES HIGIENICAS", "OUTROS", "PROJETO ESPECIAL",
    "RESIZE", "RETROFIT", "VIRADA DE BANDEIRA",
]

LOOKBACK_DAYS_DEFAULT = 15  # usado se você optar por filtrar por data
DIAGNOSTICO = True

# ======================================================================
# UI Terminal
# ======================================================================
def select_projeto_menu(opcoes: list) -> str:
    print("\n📁 Selecione o PROJETO:")
    for i, nome in enumerate(opcoes, start=1):
        print(f" {i}. {nome}")
    print(" 0. Cancelar")
    while True:
        try:
            op = int(input("Digite o número da opção: ").strip())
            if op == 0:
                return None
            if 1 <= op <= len(opcoes):
                projeto = opcoes[op - 1]
                print(f"✅ Selecionado: {projeto}")
                return projeto
        except ValueError:
            pass
        print("Opção inválida. Tente novamente.")

def ask_fornecedor_nome() -> str:
    nome = input("\nDigite o NOME exato do FORNECEDOR (ex.: LUMICENTER): ").strip()
    while not nome:
        nome = input("Digite o NOME do FORNECEDOR: ").strip()
    return nome

def ask_pasta_origem() -> str:
    while True:
        p = input("\nDigite o CAMINHO da PASTA onde estão os PDFs (ex.: C:\\Temp\\NFs): ").strip().strip('"')
        if os.path.isdir(p):
            return p
        print("❌ Caminho inválido. Tente novamente.")

def ask_yesno(msg: str, default_yes=True) -> bool:
    sufixo = "[S/n]" if default_yes else "[s/N]"
    while True:
        resp = input(f"{msg} {sufixo}: ").strip().lower()
        if not resp:
            return default_yes
        if resp in ("s", "sim", "y", "yes"):
            return True
        if resp in ("n", "nao", "não", "no"):
            return False
        print("Resposta inválida. Digite S ou N.")

def ask_int(msg: str, padrao: int) -> int:
    val = input(f"{msg} (padrão {padrao}): ").strip()
    if not val:
        return padrao
    try:
        return int(val)
    except:
        print("Valor inválido, usando padrão.")
        return padrao

# ======================================================================
# Utils locais
# ======================================================================
def _ensure_dirs(path): os.makedirs(path, exist_ok=True)

def _sanitize_filename(name: str) -> str:
    name = (name or "").strip().replace("\n", " ").replace("\r", " ")
    name = re.sub(r'[<>:"/\\|?*]', "_", name)
    name = re.sub(r"\s{2,}", " ", name)
    return name

def _md5_file(path: str) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def _normalize_text(text: str) -> str:
    if not text:
        return ""
    t = re.sub(r"[\n\r\t]+", " ", text)
    t = re.sub(r"\s{2,}", " ", t)
    return t.upper()

def _parse_brl_to_float(v):
    if v is None:
        return None
    s = str(v)
    s = re.sub(r"[Rr]\$\s*", "", s)
    s = s.replace(".", "TEMP").replace(",", ".").replace("TEMP", "")
    s = re.sub(r"[^\d\.]", "", s)
    try:
        return float(s)
    except:
        return None

# ======================================================================
# CNPJ & Loja
# ======================================================================
def _cnpj_digits(s: str) -> str:
    d = re.sub(r"\D", "", s or "")
    return d if len(d) == 14 else ""

def _cnpj_is_valid(d: str) -> bool:
    d = _cnpj_digits(d)
    if len(d) != 14 or len(set(d)) == 1:
        return False
    def dv_calc(digs, pesos):
        soma = sum(int(digs[i]) * pesos[i] for i in range(len(pesos)))
        r = soma % 11
        return '0' if r < 2 else str(11 - r)
    base = d[:12]
    dv1 = dv_calc(base, [5,4,3,2,9,8,7,6,5,4,3,2])
    dv2 = dv_calc(base + dv1, [6,5,4,3,2,9,8,7,6,5,4,3,2])
    return d[-2:] == dv1 + dv2

def _derive_loja_from_cnpj(cnpj_raw: str, origem_cnpj: str) -> tuple:
    d = _cnpj_digits(cnpj_raw)
    if not d:
        return (None, origem_cnpj)
    seq = d[8:12]  # 4 dígitos do estabelecimento
    origem_loja = origem_cnpj
    if d.startswith("04"):
        loja = "7" + seq[1:]; origem_loja += " + Regra IMIFARMA"
    elif d.startswith("06"):
        loja = seq.lstrip("0") or "0"; origem_loja += " + Regra Pague Menos"
    else:
        loja = seq.lstrip("0") or seq
    return loja, origem_loja

# ======================================================================
# Extratores
# ======================================================================
def extrair_vencimento_de_texto(texto: str) -> str or None:
    if not texto: return None
    t = _normalize_text(texto).upper()
    m = re.search(r"\b(VENC(?:IMENTO)?|VENCTO|VCTO)\b[^0-9]{0,10}(\d{2}/\d{2}/\d{4})", t)
    if m: return m.group(2)
    m2 = re.search(r"\b(\d{2}/\d{2}/\d{4})\b", t)
    return m2.group(1) if m2 else None

def _find_oc_candidates(text_upper: str) -> list:
    oc_pattern = r"\b(45\d{8,10})\b"
    cands = []
    labels = ["ORDEM DE COMPRA", "OC", "PEDIDO", "PEDIDO CLIENTE", "DADOS ADICIONAIS", "INFORMAÇÕES COMPLEMENTARES"]
    for lbl in labels:
        m_lbl = re.search(lbl, text_upper)
        if m_lbl:
            start = max(0, m_lbl.start()-2000)
            bloco = text_upper[start: m_lbl.start()+4000]
            for m in re.finditer(oc_pattern, bloco):
                cands.append((start + m.start(), m.group(1), 5))
    for m in re.finditer(oc_pattern, text_upper):
        cands.append((m.start(), m.group(1), 1))
    seen, uniq = set(), []
    for pos, oc, score in cands:
        if oc not in seen:
            seen.add(oc); uniq.append((pos, oc, score))
    return uniq

# ======================================================================
# Leitura de PDF + regras de extração (AJUSTADO)
# ======================================================================
def extrair_dados_pdf(caminho_pdf, debug=False):
    if PdfReader is None:
        return None, None, None, None, None, None, {}
    try:
        reader = PdfReader(caminho_pdf)
        texto_completo = "\n".join([p.extract_text() or "" for p in reader.pages])
        t_upper = texto_completo.upper()

        # ---------- CNPJ DESTINATÁRIO ----------
        def _pick_cnpj_dest(text_upper: str) -> str:
            cnpj_iter = list(re.finditer(r"\d{2}\.\d{3}\.\d{3}/\d{4}\-\d{2}", text_upper))
            if not cnpj_iter:
                return ""
            scored = []
            for m in cnpj_iter:
                start, end = m.start(), m.end()
                win = text_upper[max(0, start-250): min(len(text_upper), end+250)]
                score = 0
                if re.search(r"DESTINAT[ÁA]RIO|LOCAL\s+DE\s+ENTREGA|RECEBEDOR|NOME/RAZ[ÃA]O\s+SOCIAL", win):
                    score += 3
                if re.search(r"PAGUE\s+MENOS|IMIFARMA|EMPREENDE?NDIMENTOS\s+PAGUE\s+MENOS", win):
                    score += 2
                if re.search(r"EMITENTE|IDENTIFICA[ÇC][ÃA]O\s+DO\s+EMITENTE", win):
                    score -= 3
                if _cnpj_digits(m.group()).startswith("06"):
                    score += 1
                scored.append((score, m.group(), start))
            scored.sort(key=lambda x: (-x[0], x[2]))
            return _cnpj_digits(scored[0][1]) if scored else ""

        cnpj_dest = _pick_cnpj_dest(t_upper)
        loja, origem_loja = _derive_loja_from_cnpj(cnpj_dest, "CNPJ")

        # ---------- Fallback por 'LJ' no texto ----------
        m_lj_any = re.search(r"\b(?:LOJA|LJ|FL)\b[^\d]{0,10}(\d{1,5})\b", t_upper)
        if not loja and m_lj_any:
            loja = m_lj_any.group(1)
            origem_loja = (origem_loja or "CNPJ") + " + Fallback LJ"
        else:
            m_lj2 = re.search(r"\bLJ[^\d]{0,2}(\d{2,5})\b", t_upper)
            if loja and loja.isdigit() and int(loja) < 10 and m_lj2:
                loja = m_lj2.group(1)
                origem_loja = (origem_loja or "CNPJ") + " + Fallback LJ"

        # ---------- NÚMERO DA NF (AJUSTADO PARA PREFEITURA SP) ----------
        nf = None
        
        # 1) Padrão específico para NFS-e da Prefeitura de São Paulo (Extração PyPDF2)
        # O número costuma vir colado após a data de emissão no texto extraído
        m_sp_data = re.search(r"EMITIDO\s+EM\s+\d{2}/\d{2}/\d{4}(\d+)", t_upper)
        if m_sp_data:
            nf = m_sp_data.group(1)
            
        # 2) Padrão "Número da Nota" com quebra de linha ou espaço
        if not nf:
            m_sp = re.search(r"N[ÚU]MERO\s+DA\s+NOTA\s*[\r\n\s]*(\d+)", t_upper)
            if m_sp:
                nf = m_sp.group(1)
            
        # 3) Padrão com Inscrição Municipal
        if not nf:
            m_insc = re.search(r"INSCRIÇÃO\s+MUNICIPAL:\s*\d{8,}(\d{4,7})\b", t_upper)
            if m_insc:
                nf = m_insc.group(1)
                
        # 4) Rótulos comuns
        if not nf:
            padroes = [
                r"N[ÚU]MERO\s+DA\s+NOTA[\r\n\s:]*(\d+)",
                r"N[ÚU]MERO\s+DA\s+NFS?-?E[\r\n\s:]*(\d+)",
                r"NF-E\s+N[º°\.\s]*(\d+)"
            ]
            for p in padroes:
                for m in re.finditer(p, t_upper):
                    cand = m.group(1)
                    contexto = t_upper[max(0, m.start()-5):m.end()+20]
                    if "(98)" in contexto or "TELEFONE" in contexto:
                        continue
                    if 1 < len(cand) < 10:
                        nf = cand
                        break
                if nf:
                    break

        # --- OUTROS CAMPOS ---
        serie = (re.search(r"S[ÉE]RIE[^\dA-Z]{0,10}(\d+|[A-Z]+)\b", t_upper) or [None, None])[1]
        chave = (re.search(r"(\d{4}\s\d{4}\s\d{4}\s\d{4}\s\d{4}\s\d{4}\s\d{4}\s\d{4}\s\d{4}\s\d{4}\s\d{4})", t_upper) or [None, None])[1]
        if chave: chave = chave.replace(" ", "")
        
        valor = (re.search(r"VALOR\s+TOTAL\s+DO\s+SERVI[ÇC]O\s*=\s*R\$\s*([\d\.,]+)", t_upper) or [None, None])[1]
        oc_cands = _find_oc_candidates(t_upper)
        oc = max(oc_cands, key=lambda x: x[2])[1] if oc_cands else None
        data_emissao = (re.search(r"(?:DATA\s*(?:DE\s*|E\s*HORA\s*(?:DA\s*|DE\s*))?EMISS[ÃA]O)[^\d]{0,60}(\d{2}/\d{2}/\d{4})", t_upper) or [None, None])[1]

        if DIAGNOSTICO:
            print(f" - NF: {nf or 'N/D'} | CNPJ Destinatário: {cnpj_dest or 'N/D'} | Loja derivada: {loja or 'N/D'} | Origem Loja: {origem_loja or 'N/D'}")

        extras = {
            "Valor Total": valor,
            "Valor Total Num": _parse_brl_to_float(valor),
            "ordem_compra": oc,
            "pedido": oc,
            "data_emissao": data_emissao,
            "destinatario_cnpj": cnpj_dest
        }
        return nf, loja, "PDF", origem_loja, serie, chave, extras
    except Exception as e:
        if debug: print(f"Erro ao ler {caminho_pdf}: {e}")
        return None, None, None, None, None, None, {}

# ======================================================================
# Listagem de PDFs a partir de uma pasta
# ======================================================================
def listar_pdfs(pasta_origem: str, recursivo=True, filtrar_por_data=True, lookback_days=LOOKBACK_DAYS_DEFAULT):
    pdfs = []
    dt_limite = dt.datetime.now() - dt.timedelta(days=lookback_days)
    def dentro_periodo(path):
        if not filtrar_por_data:
            return True
        try:
            mtime = dt.datetime.fromtimestamp(os.path.getmtime(path))
            return mtime >= dt_limite
        except:
            return True

    if recursivo:
        for root, _, files in os.walk(pasta_origem):
            for fn in files:
                if fn.lower().endswith(".pdf"):
                    full = os.path.join(root, fn)
                    if dentro_periodo(full):
                        pdfs.append(full)
    else:
        for fn in os.listdir(pasta_origem):
            full = os.path.join(pasta_origem, fn)
            if os.path.isfile(full) and fn.lower().endswith(".pdf") and dentro_periodo(full):
                pdfs.append(full)
    return pdfs

# ======================================================================
# MAIN
# ======================================================================
def main():
    # 1) Seleções básicas
    projeto = select_projeto_menu(PROJETOS)
    if not projeto:
        return
    fornecedor = ask_fornecedor_nome()

    # 2) Pastas
    pasta_origem = ask_pasta_origem()
    recursivo = ask_yesno("Varrer SUBPASTAS?", default_yes=True)
    usar_filtro_data = ask_yesno(f"Filtrar pelos últimos {LOOKBACK_DAYS_DEFAULT} dias?", default_yes=True)
    lookback_days = LOOKBACK_DAYS_DEFAULT if usar_filtro_data else 99999

    pasta_projeto = os.path.join(BASE_PROJETOS, projeto)
    pasta_fornecedor = os.path.join(pasta_projeto, fornecedor)
    if not os.path.isdir(pasta_projeto):
        print(f"❌ Pasta do projeto não existe: {pasta_projeto}")
        return
    _ensure_dirs(pasta_fornecedor)

    # 3) Coleta PDFs na pasta informada
    pdf_paths = listar_pdfs(pasta_origem, recursivo=recursivo, filtrar_por_data=usar_filtro_data, lookback_days=lookback_days)
    print(f"\n📥 PDFs encontrados: {len(pdf_paths)}")
    if DIAGNOSTICO and pdf_paths[:5]:
        print("Exemplos:", *pdf_paths[:5], sep="\n - ")

    # 4) Processamento
    base_registros = []
    for src_path in pdf_paths:
        try:
            nf, loja, _, origem_lo, serie, chave, extras = extrair_dados_pdf(src_path)

            novo_nome = f"NF {nf} - LJ {loja}.pdf" if nf and loja else f"NF {nf or 'SEM_NUM'}.pdf"
            novo_nome = _sanitize_filename(novo_nome)
            destino = os.path.join(pasta_fornecedor, novo_nome)

            status = "OK"
            try:
                if os.path.exists(destino):
                    # Evita sobrescrever: acrescenta hash curto
                    hash5 = _md5_file(src_path)[:5]
                    base, ext = os.path.splitext(novo_nome)
                    destino = os.path.join(pasta_fornecedor, f"{base}_{hash5}{ext}")
                shutil.move(src_path, destino)
            except Exception as e:
                status = f"Erro ao mover: {e}"

            base_registros.append({
                "Projeto": projeto,
                "Fornecedor": fornecedor,
                "NF": nf,
                "Loja": loja,
                "Serie": serie,
                "Chave": chave,
                "Valor Total": extras.get("Valor Total"),
                "Valor Total Num": extras.get("Valor Total Num"),
                "Pedido": extras.get("pedido"),
                "Data Emissao": extras.get("data_emissao"),
                "CNPJ Destinatario": extras.get("destinatario_cnpj"),
                "OrigemArquivo": src_path,
                "DestinoArquivo": destino,
                "Status": status
            })
        except Exception as e:
            base_registros.append({
                "Projeto": projeto, "Fornecedor": fornecedor, "NF": None, "Loja": None,
                "Serie": None, "Chave": None, "Valor Total": None, "Valor Total Num": None,
                "Pedido": None, "Data Emissao": None, "CNPJ Destinatario": None,
                "OrigemArquivo": src_path, "DestinoArquivo": None, "Status": f"Erro geral: {e}"
            })

    # 5) Exporta Excel
    if base_registros and pd:
        df = pd.DataFrame(base_registros)
        hoje = dt.datetime.now().strftime("%Y-%m-%d")
        excel_path = os.path.join(pasta_projeto, f"BASE_NFs_{fornecedor}_{hoje}.xlsx")
        try:
            df.to_excel(excel_path, index=False)
            print(f"\n✅ Excel gerado: {excel_path}")
        except Exception as e:
            print(f"❌ Falha ao salvar Excel: {e}")
    else:
        print("⚠️ Nenhum registro para exportar ou pandas não instalado.")

    # 6) Resumo
    ok = sum(1 for r in base_registros if r.get("Status") == "OK")
    falhas = sum(1 for r in base_registros if r.get("Status") != "OK")
    print(f"\n📊 Resumo: OK={ok}  Falhas={falhas}  Total={len(base_registros)}")
    if falhas:
        print("Dicas:")
        print(" - Verifique permissões da pasta de destino.")
        print(" - Feche arquivos PDF abertos para permitir o move.")
        print(" - Alguns PDFs podem não ter texto selecionável (PDF imagem).")

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\nInterrompido pelo usuário.")



📁 Selecione o PROJETO:
 1. ESPECIAL
 2. LAUDO DE RISCO
 3. LAUDO DE RISCO CD
 4. LAUDO DE RISCO MATRIZ
 5. LOJAS NOVAS
 6. MISSOES HIGIENICAS
 7. OUTROS
 8. PROJETO ESPECIAL
 9. RESIZE
 10. RETROFIT
 11. VIRADA DE BANDEIRA
 0. Cancelar


Digite o número da opção:  5


✅ Selecionado: LOJAS NOVAS



Digite o NOME exato do FORNECEDOR (ex.: LUMICENTER):  HTB

Digite o CAMINHO da PASTA onde estão os PDFs (ex.: C:\Temp\NFs):  C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\NFS\2025\LOJAS NOVAS\HTB\24.03.2026
Varrer SUBPASTAS? [S/n]:  s
Filtrar pelos últimos 15 dias? [S/n]:  s



📥 PDFs encontrados: 6
Exemplos:
 - C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\NFS\2025\LOJAS NOVAS\HTB\24.03.2026\NF SEM_NUM.pdf
 - C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\NFS\2025\LOJAS NOVAS\HTB\24.03.2026\NF SEM_NUM_35820.pdf
 - C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\NFS\2025\LOJAS NOVAS\HTB\24.03.2026\NF SEM_NUM_3ecbd.pdf
 - C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\NFS\2025\LOJAS NOVAS\HTB\24.03.2026\NF SEM_NUM_6b568.pdf
 - C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\NFS\2025\LOJAS NOVAS\HTB\24.03.2026\NF SEM_NUM_a9719.pdf
 - NF: 00015706 | CNPJ Destinatário: 06626253133461 | Loja derivada: 1334 | Origem Loja: CNPJ + Regra Pague Menos
 - NF: 00015711 | CNPJ Destinatário: 06626253116532 | Loja derivada: 1165 | Origem Loja: CNPJ + Regra Pague Menos
 - NF: 00015708 | CNPJ Destinatário: 06626253149112 | Loja derivada: 1491 | Origem Loja: CNPJ + Regra Pague Menos
 - NF: 00015710 | CNP

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import os, re, sys, hashlib, shutil, datetime as dt

# --------- Dependências críticas ---------
missing = []
try:
    import pandas as pd
except Exception:
    pd = None
    missing.append("pandas")
try:
    from PyPDF2 import PdfReader
except Exception:
    PdfReader = None
    missing.append("PyPDF2")
try:
    import openpyxl  # apenas para o writer do pandas
except Exception:
    missing.append("openpyxl")

if missing:
    print("⚠️ Dependências ausentes:", ", ".join(missing))
    print("   Instale no Anaconda Prompt / terminal:")
    if "pandas" in missing: print("   pip install pandas")
    if "openpyxl" in missing: print("   pip install openpyxl")
    if "PyPDF2" in missing: print("   pip install PyPDF2")

# ======================================================================
# Parâmetros fixos do seu ambiente
# ======================================================================
BASE_PROJETOS = r"C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\NFS\2025"

PROJETOS = [
    "ESPECIAL", "LAUDO DE RISCO", "LAUDO DE RISCO CD", "LAUDO DE RISCO MATRIZ",
    "LOJAS NOVAS", "MISSOES HIGIENICAS", "OUTROS", "PROJETO ESPECIAL",
    "RESIZE", "RETROFIT", "VIRADA DE BANDEIRA",
]

LOOKBACK_DAYS_DEFAULT = 15  # usado se você optar por filtrar por data
DIAGNOSTICO = True

# ======================================================================
# UI Terminal
# ======================================================================
def select_projeto_menu(opcoes: list) -> str:
    print("\n📁 Selecione o PROJETO:")
    for i, nome in enumerate(opcoes, start=1):
        print(f" {i}. {nome}")
    print(" 0. Cancelar")
    while True:
        try:
            op = int(input("Digite o número da opção: ").strip())
            if op == 0:
                return None
            if 1 <= op <= len(opcoes):
                projeto = opcoes[op - 1]
                print(f"✅ Selecionado: {projeto}")
                return projeto
        except ValueError:
            pass
        print("Opção inválida. Tente novamente.")

def ask_fornecedor_nome() -> str:
    nome = input("\nDigite o NOME exato do FORNECEDOR (ex.: LUMICENTER): ").strip()
    while not nome:
        nome = input("Digite o NOME do FORNECEDOR: ").strip()
    return nome

def ask_pasta_origem() -> str:
    while True:
        p = input("\nDigite o CAMINHO da PASTA onde estão os PDFs (ex.: C:\\Temp\\NFs): ").strip().strip('"')
        if os.path.isdir(p):
            return p
        print("❌ Caminho inválido. Tente novamente.")

def ask_yesno(msg: str, default_yes=True) -> bool:
    sufixo = "[S/n]" if default_yes else "[s/N]"
    while True:
        resp = input(f"{msg} {sufixo}: ").strip().lower()
        if not resp:
            return default_yes
        if resp in ("s", "sim", "y", "yes"):
            return True
        if resp in ("n", "nao", "não", "no"):
            return False
        print("Resposta inválida. Digite S ou N.")

# ======================================================================
# Utils locais
# ======================================================================
def _ensure_dirs(path): os.makedirs(path, exist_ok=True)

def _sanitize_filename(name: str) -> str:
    name = (name or "").strip().replace("\n", " ").replace("\r", " ")
    name = re.sub(r'[<>:"/\\|?*]', "_", name)
    name = re.sub(r"\s{2,}", " ", name)
    return name

def _md5_file(path: str) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def _normalize_text(text: str) -> str:
    if not text:
        return ""
    t = re.sub(r"[\n\r\t]+", " ", text)
    t = re.sub(r"\s{2,}", " ", t)
    return t.upper()

def _parse_brl_to_float(v):
    if v is None:
        return None
    s = str(v)
    s = re.sub(r"[Rr]\$\s*", "", s)
    s = s.replace(".", "TEMP").replace(",", ".").replace("TEMP", "")
    s = re.sub(r"[^\d\.]", "", s)
    try:
        return float(s)
    except:
        return None

# ======================================================================
# CNPJ & Loja
# ======================================================================
def _cnpj_digits(s: str) -> str:
    d = re.sub(r"\D", "", s or "")
    return d if len(d) == 14 else ""

def _derive_loja_from_cnpj(cnpj_raw: str, origem_cnpj: str) -> tuple:
    d = _cnpj_digits(cnpj_raw)
    if not d:
        return (None, origem_cnpj)
    seq = d[8:12]  # 4 dígitos do estabelecimento
    origem_loja = origem_cnpj
    if d.startswith("04"):
        loja = "7" + seq[1:]; origem_loja += " + Regra IMIFARMA"
    elif d.startswith("06"):
        loja = seq.lstrip("0") or "0"; origem_loja += " + Regra Pague Menos"
    else:
        loja = seq.lstrip("0") or seq
    return loja, origem_loja

# ======================================================================
# Extratores
# ======================================================================
def _find_oc_candidates(text_upper: str) -> list:
    oc_pattern = r"\b(45\d{8,10})\b"
    cands = []
    labels = ["ORDEM DE COMPRA", "OC", "PEDIDO", "PEDIDO CLIENTE", "DADOS ADICIONAIS", "INFORMAÇÕES COMPLEMENTARES"]
    for lbl in labels:
        m_lbl = re.search(lbl, text_upper)
        if m_lbl:
            start = max(0, m_lbl.start()-2000)
            bloco = text_upper[start: m_lbl.start()+4000]
            for m in re.finditer(oc_pattern, bloco):
                cands.append((start + m.start(), m.group(1), 5))
    for m in re.finditer(oc_pattern, text_upper):
        cands.append((m.start(), m.group(1), 1))
    seen, uniq = set(), []
    for pos, oc, score in cands:
        if oc not in seen:
            seen.add(oc); uniq.append((pos, oc, score))
    return uniq

def extrair_dados_pdf(caminho_pdf, debug=False):
    if PdfReader is None:
        return None, None, None, None, None, None, {}
    try:
        reader = PdfReader(caminho_pdf)
        texto_completo = "\n".join([p.extract_text() or "" for p in reader.pages])
        t_upper = texto_completo.upper()

        # ---------- CNPJ DESTINATÁRIO ----------
        def _pick_cnpj_dest(text_upper: str) -> str:
            cnpj_iter = list(re.finditer(r"\d{2}\.\d{3}\.\d{3}/\d{4}\-\d{2}", text_upper))
            if not cnpj_iter:
                return ""
            scored = []
            for m in cnpj_iter:
                start, end = m.start(), m.end()
                win = text_upper[max(0, start-250): min(len(text_upper), end+250)]
                score = 0
                if re.search(r"DESTINAT[ÁA]RIO|LOCAL\s+DE\s+ENTREGA|RECEBEDOR|NOME/RAZ[ÃA]O\s+SOCIAL", win):
                    score += 3
                if re.search(r"PAGUE\s+MENOS|IMIFARMA|EMPREENDE?NDIMENTOS\s+PAGUE\s+MENOS", win):
                    score += 2
                if re.search(r"EMITENTE|IDENTIFICA[ÇC][ÃA]O\s+DO\s+EMITENTE", win):
                    score -= 3
                if _cnpj_digits(m.group()).startswith("06"):
                    score += 1
                scored.append((score, m.group(), start))
            scored.sort(key=lambda x: (-x[0], x[2]))
            return _cnpj_digits(scored[0][1]) if scored else ""

        cnpj_dest = _pick_cnpj_dest(t_upper)
        loja, origem_loja = _derive_loja_from_cnpj(cnpj_dest, "CNPJ")

        # ---------- Fallback por 'LJ' no texto ----------
        m_lj_any = re.search(r"\b(?:LOJA|LJ|FL)\b[^\d]{0,10}(\d{1,5})\b", t_upper)
        if not loja and m_lj_any:
            loja = m_lj_any.group(1)
            origem_loja = (origem_loja or "CNPJ") + " + Fallback LJ"

        # ---------- NÚMERO DA NF (AJUSTADO) ----------
        nf = None
        
        # 1) Padrão específico para NFS-e da Prefeitura de São Paulo
        m_sp_data = re.search(r"EMITIDO\s+EM\s+\d{2}/\d{2}/\d{4}(\d+)", t_upper)
        if m_sp_data:
            nf = m_sp_data.group(1)
            
        # 2) Padrão "Número da Nota"
        if not nf:
            m_sp = re.search(r"N[ÚU]MERO\s+DA\s+NOTA\s*[\r\n\s]*(\d+)", t_upper)
            if m_sp:
                nf = m_sp.group(1)
            
        # 3) Padrão com Inscrição Municipal
        if not nf:
            m_insc = re.search(r"INSCRIÇÃO\s+MUNICIPAL:\s*\d{8,}(\d{4,7})\b", t_upper)
            if m_insc:
                nf = m_insc.group(1)
                
        # 4) Rótulos comuns
        if not nf:
            padroes = [
                r"N[ÚU]MERO\s+DA\s+NOTA[\r\n\s:]*(\d+)",
                r"N[ÚU]MERO\s+DA\s+NFS?-?E[\r\n\s:]*(\d+)",
                r"NF-E\s+N[º°\.\s]*(\d+)"
            ]
            for p in padroes:
                for m in re.finditer(p, t_upper):
                    cand = m.group(1)
                    contexto = t_upper[max(0, m.start()-5):m.end()+20]
                    if "(98)" in contexto or "TELEFONE" in contexto:
                        continue
                    if 1 < len(cand) < 10:
                        nf = cand
                        break
                if nf:
                    break

        # REMOVER ZEROS À ESQUERDA DA NF
        if nf:
            nf = nf.lstrip('0') or '0'

        # --- OUTROS CAMPOS ---
        serie = (re.search(r"S[ÉE]RIE[^\dA-Z]{0,10}(\d+|[A-Z]+)\b", t_upper) or [None, None])[1]
        chave = (re.search(r"(\d{4}\s\d{4}\s\d{4}\s\d{4}\s\d{4}\s\d{4}\s\d{4}\s\d{4}\s\d{4}\s\d{4}\s\d{4})", t_upper) or [None, None])[1]
        if chave: chave = chave.replace(" ", "")
        
        valor = (re.search(r"VALOR\s+TOTAL\s+DO\s+SERVI[ÇC]O\s*=\s*R\$\s*([\d\.,]+)", t_upper) or [None, None])[1]
        if not valor:
            valor = (re.search(r"VALOR\s*TOTAL[^\d]{0,20}([\d\.,]{4,})", t_upper) or [None, None])[1]
            
        oc_cands = _find_oc_candidates(t_upper)
        oc = max(oc_cands, key=lambda x: x[2])[1] if oc_cands else None
        data_emissao = (re.search(r"(?:DATA\s*(?:DE\s*|E\s*HORA\s*(?:DA\s*|DE\s*))?EMISS[ÃA]O)[^\d]{0,60}(\d{2}/\d{2}/\d{4})", t_upper) or [None, None])[1]

        if DIAGNOSTICO:
            print(f" - NF: {nf or 'N/D'} | Loja: {loja or 'N/D'} | CNPJ: {cnpj_dest or 'N/D'}")

        extras = {
            "Valor Total": valor,
            "Valor Total Num": _parse_brl_to_float(valor),
            "ordem_compra": oc,
            "pedido": oc,
            "data_emissao": data_emissao,
            "destinatario_cnpj": cnpj_dest
        }
        return nf, loja, "PDF", origem_loja, serie, chave, extras
    except Exception as e:
        if debug: print(f"Erro ao ler {caminho_pdf}: {e}")
        return None, None, None, None, None, None, {}

# ======================================================================
# Listagem de PDFs
# ======================================================================
def listar_pdfs(pasta_origem: str, recursivo=True, filtrar_por_data=True, lookback_days=LOOKBACK_DAYS_DEFAULT):
    pdfs = []
    dt_limite = dt.datetime.now() - dt.timedelta(days=lookback_days)
    if recursivo:
        for root, _, files in os.walk(pasta_origem):
            for fn in files:
                if fn.lower().endswith(".pdf"):
                    full = os.path.join(root, fn)
                    try:
                        if not filtrar_por_data or dt.datetime.fromtimestamp(os.path.getmtime(full)) >= dt_limite:
                            pdfs.append(full)
                    except: pass
    else:
        for fn in os.listdir(pasta_origem):
            full = os.path.join(pasta_origem, fn)
            if os.path.isfile(full) and fn.lower().endswith(".pdf"):
                try:
                    if not filtrar_por_data or dt.datetime.fromtimestamp(os.path.getmtime(full)) >= dt_limite:
                        pdfs.append(full)
                except: pass
    return pdfs

# ======================================================================
# MAIN
# ======================================================================
def main():
    projeto = select_projeto_menu(PROJETOS)
    if not projeto: return
    fornecedor = ask_fornecedor_nome()

    pasta_origem = ask_pasta_origem()
    recursivo = ask_yesno("Varrer SUBPASTAS?", default_yes=True)
    usar_filtro_data = ask_yesno(f"Filtrar pelos últimos {LOOKBACK_DAYS_DEFAULT} dias?", default_yes=True)
    lookback_days = LOOKBACK_DAYS_DEFAULT if usar_filtro_data else 99999

    pasta_projeto = os.path.join(BASE_PROJETOS, projeto)
    pasta_fornecedor = os.path.join(pasta_projeto, fornecedor)
    if not os.path.isdir(pasta_projeto):
        print(f"❌ Pasta do projeto não existe: {pasta_projeto}"); return
    _ensure_dirs(pasta_fornecedor)

    pdf_paths = listar_pdfs(pasta_origem, recursivo=recursivo, filtrar_por_data=usar_filtro_data, lookback_days=lookback_days)
    print(f"\n📥 PDFs encontrados: {len(pdf_paths)}")

    base_registros = []
    for src_path in pdf_paths:
        try:
            nf, loja, _, origem_lo, serie, chave, extras = extrair_dados_pdf(src_path)

            novo_nome = f"NF {nf} - LJ {loja}.pdf" if nf and loja else f"NF {nf or 'SEM_NUM'}.pdf"
            novo_nome = _sanitize_filename(novo_nome)
            destino = os.path.join(pasta_fornecedor, novo_nome)

            status = "OK"
            try:
                if os.path.exists(destino):
                    hash5 = _md5_file(src_path)[:5]
                    base, ext = os.path.splitext(novo_nome)
                    destino = os.path.join(pasta_fornecedor, f"{base}_{hash5}{ext}")
                shutil.move(src_path, destino)
            except Exception as e:
                status = f"Erro ao mover: {e}"

            base_registros.append({
                "Projeto": projeto,
                "Fornecedor": fornecedor,
                "NF": nf,
                "Loja": loja,
                "Serie": serie,
                "Chave": chave,
                "Valor Total": extras.get("Valor Total"),
                "Valor Total Num": extras.get("Valor Total Num"),
                "Pedido": extras.get("pedido"),
                "Data Emissao": extras.get("data_emissao"),
                "CNPJ Destinatario": extras.get("destinatario_cnpj"),
                "OrigemArquivo": src_path,
                "DestinoArquivo": destino,
                "Status": status
            })
        except Exception as e:
            base_registros.append({
                "Projeto": projeto, "Fornecedor": fornecedor, "NF": None, "Loja": None,
                "Status": f"Erro geral: {e}", "OrigemArquivo": src_path
            })

    # EXPORTAÇÃO EXCEL (INDIVIDUAL E RESUMO)
    if base_registros and pd:
        df = pd.DataFrame(base_registros)
        hoje = dt.datetime.now().strftime("%Y-%m-%d")
        
        # 1. Excel Individual do Fornecedor
        excel_path = os.path.join(pasta_projeto, f"BASE_NFs_{fornecedor}_{hoje}.xlsx")
        df.to_excel(excel_path, index=False)
        print(f"\n✅ Excel do fornecedor gerado: {excel_path}")
        
        # 2. Excel de Resumo Consolidado (Lógica do segundo código)
        resumo_path = os.path.join(pasta_projeto, f"RESUMO_CONSOLIDADO_{projeto}.xlsx")
        try:
            if os.path.exists(resumo_path):
                df_antigo = pd.read_excel(resumo_path)
                df_final = pd.concat([df_antigo, df], ignore_index=True).drop_duplicates(subset=["NF", "Fornecedor", "Loja"], keep="last")
            else:
                df_final = df
            df_final.to_excel(resumo_path, index=False)
            print(f"✅ Resumo consolidado atualizado: {resumo_path}")
        except Exception as e:
            print(f"⚠️ Erro ao atualizar resumo: {e}")

    print(f"\n📊 Resumo: OK={sum(1 for r in base_registros if r.get('Status') == 'OK')}  Total={len(base_registros)}")

if __name__ == "__main__":
    main()
